In [ ]:
# Determining the optimal number of hidden layers and neurons for an
# Artificial Neural Network (ANN) is a crucial step in the model development 
# process. The architecture of the ANN can significantly impact its performance,
# and finding the right balance between complexity and generalization is key.
# Here are some guidelines to help you determine the optimal number of hidden 
# layers and neurons:

# This can be challenging and often requires experimentation. However, there 
# are some guidelines and methods that can help you in making an informed decision : 

# 1. Start Simple: Begin with a simple architecture and gradually increase complexity if needed. 
# 2. Grid Search/Random Search: Use grid search or random search to try different architectures.
# 3. Cross-Validation: Use cross-validation to evaluate the performance of different archtitectures. 
# 4. Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points such as: 
#     - The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer. 
#     - A common practice is to start with 1-2 hidden layers. 



In [1]:
import tensorflow as tf
print(tf.__version__)

2.15.0


In [2]:
from scikeras.wrappers import KerasClassifier
print("Working")

Working


In [3]:
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV 
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder 
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier 
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import EarlyStopping
import pickle 





In [5]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

x = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, Y_train, Y_test = train_test_split(x,y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Save encoders and scaler for later use 
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)





In [6]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32, layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu', input_shape=(X_train.shape[1])))
    
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='admin',loss='binary_crossentropy',metrics=['accuracy'])   

    return model 

 

In [7]:
## Create a Keras classifier 
model=KerasClassifier(build_fn=create_model,epochs=50,batch_size=10,verbose=0)



In [11]:
# Define the grid search parameters 
# param_grid ={
#     'neurons': [16,32,64,128],
#     'layers': [1,2],
#     'epochs': [50,100]
# }
param_grid = {
    'model__neurons': [16, 32, 64, 128],
    'model__layers': [1, 2],
    'epochs': [50, 100]
}

In [13]:
def create_model(neurons=32, layers=1):
    model = Sequential()

    model.add(Dense(neurons, activation='relu', input_dim=X_train.shape[1]))

    for _ in range(layers):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [15]:
model = KerasClassifier(
    model=create_model,
    verbose=0
)

In [ ]:
# Perform grid search 
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1,cv=3,verbose=1)
grid_result = grid.fit(X_train, Y_train)

## Print the best parameters 
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))




Fitting 3 folds for each of 16 candidates, totalling 48 fits
